In [22]:
#2021.10.13. WED
#Team_SmokeFree

## Data Preprocessing, XY Extraction Using Crawling
#00. 패키지 호출하기.
import warnings
import pandas as pd 
import numpy as np 
import time
import re 
from selenium import webdriver

#00-1. warning message ignore 
warnings.filterwarnings(action='ignore')

#01. 법정동을 행정동으로 변환하기. 
#(1) 데이터셋 불러오기. 
SS_data = pd.read_excel('../data/01_Feature_Data/서울특별시_담배소매업_pp.xlsx')

#(2) 처리 결과 확인하기. 
SS_data

,INDEX,자치구,행정동_코드,행정동,사업장명,소재지_지번_주소,소재지_도로명_주소,위도,경도
0,0,금천구,1.154571e+09,시흥제5동,지에스25 금천다온,서울특별시 금천구 시흥동 923번지,서울특별시 금천구 독산로6가길 18 1층 일부호 (시흥동),37.448859,126.905963
1,1,용산구,1.117062e+09,한강로동,씨유 선인상가점,NaN,서울특별시 용산구 새창로 181 21동 2018호 (한강로2가 선인상가),37.532717,126.965178
2,2,동대문구,1.123070e+09,청량리동,지에스(GS)25영휘원점,서울특별시 동대문구 청량리동 199번지 121호,서울특별시 동대문구 홍릉로 78-1 (청량리동),37.587201,127.043224
3,3,동작구,1.159065e+09,사당제5동,이마트24동작골든포레점,서울특별시 동작구 사당동 181번지 사당 롯데캐슬 골든포레,서울특별시 동작구 사당로 90 상가1동 105호 (사당동 사당 롯데캐슬 골든포레),37.492336,126.962448
4,4,광진구,1.121574e+09,중곡제1동,지에스25 중곡희망점,NaN,서울특별시 광진구 동일로68길 49 1층 (중곡동 월드빌),37.562168,127.078546
...,...,...,...,...,...,...,...,...,...
20494,20494,강북구,1.130554e+09,NaN,씨유 강북오패산로점,서울특별시 강북구 미아동 245-32,서울특별시 강북구 오패산로 271 1층 (미아동),37.625541,127.029678
20495,20495,강서구,1.150054e+09,NaN,GS25 신월초교점,서울특별시 강서구 화곡동 1081-7,서울특별시 강서구 화곡로24길 14(화곡동),37.540314,126.838198
20496,20496,은평구,1.138062e+09,NaN,럭키스토어,서울특별시 은평구 역촌동 2-38 SM 벨리체,서울특별시 은평구 서오릉로 107 1층 104호 (역촌동 SM 벨리체),37.609286,126.919912
20497,20497,동대문구,1.123065e+09,NaN,온오프,서울특별시 동대문구 장안동 414-8,서울특별시 동대문구 천호대로 365(장안동),37.562558,127.059736


In [24]:
#(3) 좌표가 없는 행 추출하기.
SS_data_no_xy = SS_data[SS_data['위도']=='ERROR']
SS_data_no_xy

,INDEX,자치구,행정동_코드,행정동,사업장명,소재지_지번_주소,소재지_도로명_주소,위도,경도
24,24,ERROR,0.0,NaN,은호주단,서울특별시 종로구 종로5가 398호,NaN,ERROR,ERROR
97,97,ERROR,0.0,NaN,서대문역매점,서울특별시 종로구 평동 90,NaN,ERROR,ERROR
232,232,ERROR,0.0,NaN,신생슈퍼,서울특별시 종로구 공평동 34,NaN,ERROR,ERROR
272,272,ERROR,0.0,NaN,-,서울특별시 종로구 인사동 165,NaN,ERROR,ERROR
300,300,ERROR,0.0,NaN,감사원매점(자판기),서울특별시 종로구 삼청동 산 2-26,NaN,ERROR,ERROR
...,...,...,...,...,...,...,...,...,...
17546,17546,ERROR,0.0,NaN,지에스25 천호은혜점,NaN,서울특별시 강동구 천중로32길 29,ERROR,ERROR
17655,17655,ERROR,0.0,NaN,세븐마트,서울특별시 강동구 고덕동 492,NaN,ERROR,ERROR
17669,17669,ERROR,0.0,NaN,강동서점,서울특별시 강동구 상일동 162,NaN,ERROR,ERROR
17736,17736,ERROR,0.0,NaN,가로가판점,서울특별시 강동구 상일동 328,NaN,ERROR,ERROR


In [25]:
#03. 드라이버 호출 및 크롤링 구동하기. 
#(0) 드라이버 옵션 설정하기. 
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")

#(1) 드라이버 호출하기. 
driver = webdriver.Chrome('../Module/chromedriver.exe', chrome_options = options)

#(2) 다울주소변환 사이트로 들어가기.
driver.get('http://www.dawuljuso.com/index.php')

#(3) 팝업 지우기. 
time.sleep(1)
driver.switch_to_window(driver.window_handles[1])
driver.close()

#(4) 서치 박스 지정하기. 
time.sleep(0.5)
driver.switch_to_window(driver.window_handles[0])
search_box = driver.find_element_by_xpath('//*[@id="input_juso"]')

#(5) 좌표 값 및 정제 주소값 추출하기. 
SS_data_no_xy_소재지_지번_주소   = list(SS_data_no_xy['소재지_지번_주소'].values)
SS_data_no_xy_소재지_도로명_주소 = list(SS_data_no_xy['소재지_도로명_주소'].values)
SS_data_no_xy_위도               = []
SS_data_no_xy_경도               = []
for i in range(len(SS_data_no_xy_소재지_지번_주소))                                   :
    result_x = ''
    result_y = ''
    time.sleep(1)
    # 검색어 지정 #
    try                                                                               :
        key = SS_data_no_xy_소재지_지번_주소[i]
        search_box.send_keys(key)
    except                                                                            :
        key = SS_data_no_xy_소재지_도로명_주소[i]
        search_box.send_keys(key)
    # 검색어 기반 값 호출 후 데이터 추출 #
    try                                                                               :
        driver.find_element_by_xpath('//*[@id="btnSch"]').click()
        time.sleep(0.25)
        result_xy = driver.find_element_by_css_selector('#insert_data_5').text
        result_x = re.sub('[^0-9.]','',result_xy.split(',')[0])
        result_y = re.sub('[^0-9.]','',result_xy.split(',')[1])
    except                                                                            :
        try                                                                           : 
            key = SS_data_no_xy_소재지_도로명_주소[i]
            search_box.send_keys(key)
            driver.find_element_by_xpath('//*[@id="btnSch"]').click()
            time.sleep(0.25)
            result_xy = driver.find_element_by_css_selector('#insert_data_5').text
            result_x = re.sub('[^0-9.]','',result_xy.split(',')[0])
            result_y = re.sub('[^0-9.]','',result_xy.split(',')[1])
        except                                                                        :
            result_x = 'ERROR'
            result_y = 'ERROR'
    # 리스트에 값 저장 #
    SS_data_no_xy_경도.append(result_x)
    SS_data_no_xy_위도.append(result_y)
driver.close()

#(6) 처리 결과 저장하기. 
SS_data_no_xy = pd.DataFrame({
    'INDEX'              : SS_data_no_xy['INDEX'].values,
    '자치구'             : SS_data_no_xy['자치구'].values,
    '행정동_코드'        : SS_data_no_xy['행정동_코드'].values,
    '행정동'             : SS_data_no_xy['행정동'].values,
    '사업장명'           : SS_data_no_xy['사업장명'].values,
    '소재지_지번_주소'   : SS_data_no_xy['소재지_지번_주소'].values,
    '소재지_도로명_주소' : SS_data_no_xy['소재지_도로명_주소'].values,
    '위도'               : SS_data_no_xy_위도, 
    '경도'               : SS_data_no_xy_경도
})

#(7) 처리 결과 확인하기. 
SS_data_no_xy

In [4]:
#(8) 인덱스 기반 매칭하기.
for i in range(len(SS_data)) :
    for k in range(len(SS_data_no_xy)) :
        if SS_data['INDEX'][i] == SS_data_no_xy['INDEX'][k] :
            SS_data['위도'][i] = SS_data_no_xy['위도'][k]
            SS_data['경도'][i] = SS_data_no_xy['경도'][k]

#(9) 처리 결과 확인하기. 
SS_data

,INDEX,자치구,행정동_코드,행정동,사업장명,소재지_지번_주소,소재지_도로명_주소,위도,경도
0,0,금천구,1.154571e+09,시흥제5동,지에스25 금천다온,서울특별시 금천구 시흥동 923번지,서울특별시 금천구 독산로6가길 18 1층 일부호 (시흥동),37.448859,126.905963
1,1,용산구,1.117062e+09,한강로동,씨유 선인상가점,NaN,서울특별시 용산구 새창로 181 21동 2018호 (한강로2가 선인상가),37.532717,126.965178
2,2,동대문구,1.123070e+09,청량리동,지에스(GS)25영휘원점,서울특별시 동대문구 청량리동 199번지 121호,서울특별시 동대문구 홍릉로 78-1 (청량리동),37.587201,127.043224
3,3,동작구,1.159065e+09,사당제5동,이마트24동작골든포레점,서울특별시 동작구 사당동 181번지 사당 롯데캐슬 골든포레,서울특별시 동작구 사당로 90 상가1동 105호 (사당동 사당 롯데캐슬 골든포레),37.492336,126.962448
4,4,광진구,1.121574e+09,중곡제1동,지에스25 중곡희망점,NaN,서울특별시 광진구 동일로68길 49 1층 (중곡동 월드빌),37.562168,127.078546
...,...,...,...,...,...,...,...,...,...
20496,20496,강북구,1.130554e+09,NaN,씨유 강북오패산로점,서울특별시 강북구 미아동 245-32,서울특별시 강북구 오패산로 271 1층 (미아동),37.625541,127.029678
20497,20497,강서구,1.150054e+09,NaN,GS25 신월초교점,서울특별시 강서구 화곡동 1081-7,서울특별시 강서구 화곡로24길 14(화곡동),37.540314,126.838198
20498,20498,은평구,1.138062e+09,NaN,럭키스토어,서울특별시 은평구 역촌동 2-38 SM 벨리체,서울특별시 은평구 서오릉로 107 1층 104호 (역촌동 SM 벨리체),37.609286,126.919912
20499,20499,동대문구,1.123065e+09,NaN,온오프,서울특별시 동대문구 장안동 414-8,서울특별시 동대문구 천호대로 365(장안동),37.562558,127.059736


In [5]:
#(10) 결과 저장하기. 
SS_data.to_excel('temp_file.xlsx', index=False)